# GPT-2 Large: Factual Recall Baseline

36 layers, d_model=1280, d_mlp=5120, 20 heads.
Same prompts as GPT-2 Medium baseline for cross-model comparison.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [8]:
model, info = load_model("gpt2-large")

Loaded pretrained model gpt2-large into HookedTransformer
Loaded gpt2-large on mps
  36 layers | 20 heads | d_model=1280 | d_mlp=5120 | sequential attn+MLP


In [9]:
prompts = [
    ("The capital of France is", " Paris", "paris"),
    ("The capital of Germany is", " Berlin", "berlin"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  PARIS: "The capital of France is" → " Paris"

Model: gpt2-large
Prompt: "The capital of France is"
Target: " Paris" (token 6342)
Final prediction: " a"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         also                2676           0.000034    
1         also                1877           0.000080    
2         also                1401           0.000107    
3         now                 771            0.000179    
4         not                 580            0.000229    
5         now                 584            0.000227    
6         now                 1062           0.000141    
7         reportedly          1147           0.000125    
8         reportedly          1373           0.000105    
9         now                 1039           0.000133    
10        now                 1226           0.000110    
11        now                 1671           0.000083    
12        now                 1183 

In [ ]:
for label, data in results.items():
    print(f"\n--- {label} ---")
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [ ]:
model_name = "gpt2-large"

for label, data in results.items():
    data["lens"].to_csv(f"../../results/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/{model_name}")

In [12]:
unload(model)

Model unloaded, memory cleared
